In [ ]:
# # data preparation --- data for 2021-2023, and merge with heritage data (volume)
# import pandas as pd

# # read the CSV file
# df = pd.read_csv('data_optim_add 1.csv')  #

# # 1️⃣ 找到包含 2021-2023 的 csemCur
# valid_csemCur = (
#     df[df['year'].isin([2021, 2022, 2023])]
#     .groupby('csemCur')['year']
#     .nunique()
# )

# valid_csemCur = valid_csemCur[valid_csemCur == 3].index

# # 2️⃣ 只保留这些 csemCur，并且只保留 2021-2023
# new_df = (
#     df[
#         (df['csemCur'].isin(valid_csemCur)) &
#         (df['year'].isin([2021, 2022, 2023]))
#     ]
#     .sort_values(['csemCur', 'year'])
#     .reset_index(drop=True)
# )

# # load heritage data
# donnees_patrimoniales = pd.read_csv('Donnees patrimoniales - volume.csv')

# donnees_patrimoniales['csemCur'] = pd.to_numeric(donnees_patrimoniales['csemCur'], errors='coerce')

# # merge volume column based on csemCur (left join to keep all rows in new_df)
# new_df = new_df.merge(
#     donnees_patrimoniales[['csemCur', 'Volume']], 
#     on='csemCur', 
#     how='left'
# )

# # convert csemCur back to Int
# new_df['csemCur'] = pd.to_numeric(new_df['csemCur'], errors='coerce').astype('Int64')  # use Int64 to allow for NaN values

# # save to a new CSV
# new_df.to_csv('new_df_with_volume_New.csv', index=False)

# # results
# print("New DataFrame (new_df) created successfully!")
# print(f"Shape: {new_df.shape}")
# print(f"Unique csemCur count: {new_df['csemCur'].nunique()}")
# print(new_df.head())



New DataFrame (new_df) created successfully!
Shape: (1035, 13)
Unique csemCur count: 345
      csemCur  year  delta_pressure   cr  \
0  5000195187  2021             NaN  0.0   
1  5000195187  2022             NaN  0.0   
2  5000195187  2023             NaN  0.0   
3  5000200795  2021             NaN  0.0   
4  5000200795  2022             NaN  0.0   

   compensatedRelativePressure_mafiltered  COPER  MANX     alpha  \
0                                     NaN      0     0  0.000000   
1                                3.449960      0     0  0.026605   
2                                     NaN      0     1  0.000000   
3                                     NaN      0     0  0.000000   
4                                3.623786      0     0  0.000647   

   algo_critere_exploit_B_mafiltered  algo_critere_env_B_mafiltered  \
0                              False                          False   
1                              False                          False   
2                       

In [28]:
import pandas as pd
import numpy as np
pd.set_option('display.float_format', '{:.6f}'.format)

# =============================================================================
# 0. 读取数据并加入threshold列
# =============================================================================
new_df = pd.read_csv('new_df_with_volume.csv')
# add CR_threshold column based on Pression Nominale Relative
new_df.loc[:, 'CR_threshold'] = new_df['Pression Nominale Relative'] * 0.91

# =============================================================================
# 1. 参数配置（新增成本参数）
# =============================================================================
T = 17
k = 0.1
mu = 4.2 # improvement factor of MANT
base_year = 2024
        
C_MANT  = 48000      # 触发后下一年的 action cost
V = new_df['Volume'].head(T).tolist() 
K = [10**2 * 146 * V[t] / (8.314 * 293.15) for t in range(T)]    # K[i] calculation
c_factors = [0.0952 + 0.004 * t for t in range(T)]  # [0.0952, 0.0992, ..., 0.1312]
C_leak_t = [c * 25200 for c in c_factors]  # [2399.04, 2499.84, ..., 3306.24]

# =============================================================================
# 2. 提取 2023 年 alpha
# =============================================================================
new_df['csemCur'] = new_df['csemCur'].astype(int)
# 🔥 一次性提取所有需要的列
df_alpha_2023 = new_df.groupby("csemCur")[["alpha", "Pression Nominale Relative", "CR_threshold"]].last().reset_index()
df_alpha_2023.columns = ["csemCur", "alpha_2023", "Pression Nominale Relative", "CR_threshold"]
print("2023 年 alpha 基准值：")
print(df_alpha_2023)
print(f"总计 {len(df_alpha_2023)} 个 csemCur\n")

# =============================================================================
# 3. 运行模拟（新增成本计算）
# =============================================================================
rows = []
for _, row in df_alpha_2023.iterrows():
    csemCur = int(row["csemCur"])
    alpha_2023 = row["alpha_2023"]
    pression_nominale = row["Pression Nominale Relative"]
    cr_threshold = row["CR_threshold"]
    
    alpha_current = alpha_2023 * 1.01
    triggered_previous_year = False  # 第一年无前一年触发
    
    for t in range(T):
        year = base_year + t
        
        # 计算泄漏量和间隔
        leak = (alpha_current * 365 / (pression_nominale - cr_threshold)) * K[t]
        interval = 365 / (alpha_current * 365 / (pression_nominale - cr_threshold)) if alpha_current > 0 else np.inf
        
        # 检查本年触发
        trigger_this_year = (leak >= 5) or (interval <= 60)
        
        # 计算成本
        leak_cost = leak * C_leak_t[t]  # 泄漏成本随时间变化
        action_cost = C_MANT if triggered_previous_year else 0
        total_cost = leak_cost + action_cost
        
        rows.append({
            "csemCur": csemCur,
            "year": year,
            "alpha": alpha_current,
            "leak": leak,
            "interval": interval,
            "triggered": trigger_this_year,
            "leak_cost": leak_cost,
            "action_cost": action_cost,
            "total_cost": total_cost,
        })
        
        # 更新下一年状态
        # 🔥 通用逻辑：任何年 trigger，下一年 = 当年alpha / mu（跳过增长）
        if trigger_this_year:
            alpha_next = alpha_current / mu  # 基于**当年**alpha / mu
        else:
            alpha_next = alpha_current * 1.01  # 正常增长
        alpha_current = alpha_next
        triggered_previous_year = trigger_this_year

# =============================================================================
# 4. 生成结果
# =============================================================================
sim_df = pd.DataFrame(rows)
sim_df['csemCur'] = sim_df['csemCur'].astype(int)


# =============================================================================
# 5. 成本汇总统计
# =============================================================================
print("\n=== 🚀 整合成本汇总统计 ===")

# 计算各csemCur的汇总统计
cost_summary = sim_df.groupby("csemCur")["total_cost"].sum().round(0).reset_index()
cost_summary.columns = ["csemCur", "total_T years_cost"]

leak_summary = sim_df.groupby("csemCur")["leak"].sum().round(3).reset_index()
leak_summary.columns = ["csemCur", "total_leak_kg"]

# 🔥 分开统计两个触发条件
sim_df['criterion_exploit'] = (sim_df['leak'] >= 5)  # 泄漏量触发
sim_df['criterion_environment'] = (sim_df['interval'] <= 60)  # 间隔触发

trigger_exploit = sim_df.groupby("csemCur")["criterion_exploit"].sum().astype(int).reset_index()
trigger_exploit.columns = ["csemCur", "trigger_exploit_times"]

trigger_env = sim_df.groupby("csemCur")["criterion_environment"].sum().astype(int).reset_index()
trigger_env.columns = ["csemCur", "trigger_env_times"]

# 🔥 修复：直接计算年数，不用reset_index重命名
total_years_df = sim_df.groupby("csemCur").size().reset_index(name='total_years')

# 🔥 整合所有统计到一个DataFrame
summary_df = (cost_summary.merge(leak_summary, on='csemCur', how='outer')
                       .merge(trigger_exploit, on='csemCur', how='outer')
                       .merge(trigger_env, on='csemCur', how='outer')
                       .merge(total_years_df, on='csemCur', how='outer'))

# 计算触发率
summary_df['trigger_exploit_rate'] = (summary_df['trigger_exploit_times'] / summary_df['total_years']).round(3)
summary_df['trigger_env_rate'] = (summary_df['trigger_env_times'] / summary_df['total_years']).round(3)
summary_df['total_trigger_rate'] = ((summary_df['trigger_exploit_times'] + summary_df['trigger_env_times']) / summary_df['total_years']).round(3)

print("🎯 各csemCur完整汇总统计：")
print(summary_df.round(3))

# print("\n📊 按泄漏量criterion触发次数排序：")
# print(summary_df.sort_values('trigger_exploit_times', ascending=False).round(3).head(10))

# print("\n📊 按环境criterion触发次数排序：")
# print(summary_df.sort_values('trigger_env_times', ascending=False).round(3).head(10))

print("\n📊 按泄漏量排序：")
print(summary_df.sort_values('total_leak_kg', ascending=False).round(3).head(10))

# 🔥 保存整合汇总表
# summary_df.to_csv("csemCur_summary_stats.csv", index=False)
# print("\n✅ 整合汇总表已保存：csemCur_summary_stats.csv")


# =============================================================================
# 6. 保存结果
# =============================================================================
sim_df.to_csv("simulation_with_costs.csv", index=False)
print("\n✅ Simulation completed!")
print("✅ 结果已保存：simulation_with_costs.csv")
print("✅ 前10行预览：")
print(sim_df.head(10).round({'alpha':6, 'leak':3, 'leak_cost':0, 'total_cost':0})[display_cols])


2023 年 alpha 基准值：
        csemCur  alpha_2023  Pression Nominale Relative  CR_threshold
0    5000200795    0.004915                    3.500000      3.185000
1    5000252448    0.181613                    3.500000      3.185000
2    5000252449    0.022172                    3.500000      3.185000
3    5000252450    0.018647                    3.500000      3.185000
4    5000252524    0.032852                    3.500000      3.185000
..          ...         ...                         ...           ...
590  5002343706    0.121390                    3.500000      3.185000
591  5002343707    0.063426                    3.500000      3.185000
592  5002699073    0.019229                    3.500000      3.185000
593  5002903325    0.001703                    3.500000      3.185000
594  5002903326    0.030425                    3.500000      3.185000

[595 rows x 4 columns]
总计 595 个 csemCur


=== 🚀 整合成本汇总统计 ===
🎯 各csemCur完整汇总统计：
        csemCur  total_T years_cost  total_leak_kg  trigger_ex